In [13]:
import pandas as pd
import kagglehub

# Download latest version
path = kagglehub.dataset_download("jrobischon/wikipedia-movie-plots")

from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/Hofstra/year3
%ls *.csv

#%cd /content/drive/MyDrive/TextMining/DataSets
#%ls *.csv

Using Colab cache for faster access to the 'wikipedia-movie-plots' dataset.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Hofstra/year3
'Copy of job_title_des.csv'   job_title_des.csv   wiki_movie_plots_deduped.csv


In [14]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [15]:
import numpy as np
import json
import pickle
import math
import spacy
from collections import defaultdict
import os

In [16]:
#test spacy is working
nlp = spacy.load('en_core_web_sm')
print("spaCy loaded")

spaCy loaded


1.1
Use the same movie set as in HW 2 (from 1980 to now) and the same inverted index (save the inverted index in a json file so you can load it later without doing the nlp steps).

In [17]:

with open('inverted_index.pkl', 'rb') as files:
  #opened pickle file from hw2 in read binary mode, load to dictionary
  inverted_index = defaultdict(dict,pickle.load(files))

with open('documents.pkl', 'rb') as files:
  documents = pickle.load(files)

with open('inverted_index.json', 'w') as files:
  #opened json file of inverted index in write mode to be used later
  json.dump(dict(inverted_index),files)

print(f"Size of inverted index: {len(inverted_index)}\nSize of Documents: {len(documents)}")

queries = [
    "scary movies to watch at night",
    "romantic comedy movies to watch for fun",
    "worst action movies of all time"
]

Size of inverted index: 179862
Size of Documents: 19994


In [18]:
#Load movie data
print("Loading movie dataset...")
movies_data = pd.read_csv('wiki_movie_plots_deduped.csv')
post1980 = movies_data[movies_data["Release Year"] > 1980]
print(f"Loaded {len(post1980)} movies from 1980+")

Loading movie dataset...
Loaded 19994 movies from 1980+


In [19]:
#HW 2 functions

from pprint import pp
from collections import defaultdict
import spacy

nlp = spacy.load('en_core_web_sm')

def compute_freq(lst):
    freq = defaultdict(int)
    for term in lst:
        freq[term] += 1
    return freq

def extract_terms(text):
    doc = nlp(text)
    out_terms = []

    #extract regular tokens lemmitized
    for token in doc:
        if token.is_alpha and not token.is_stop:
            out_terms.append(token.lemma_.lower())

    #extract multi-word named entities
    for ent in doc.ents:
        if ent.label_ in ['PERSON', 'LOC', 'ORG', 'GPE'] and len(ent.text.split()) > 1:
            entity_term = ent.text.lower().replace(' ', '_')
            out_terms.append(entity_term)

    return out_terms


def build_inverted_index(documents):
    inverted_index = defaultdict(dict)
    for doc_id, doc in enumerate(documents):
        terms_doc = extract_terms(doc)
        dict_doc = compute_freq(terms_doc)
        for term_doc, term_freq in dict_doc.items():
            inverted_index[term_doc][doc_id] = term_freq
    return inverted_index

def dist_query_docs(query_term_freq, inverted_index, topk=5):
    dot_product = defaultdict(int)
    for query_term, query_tf in query_term_freq.items():
        if query_term in inverted_index:
            dict_term_inverted_index = inverted_index[query_term]
            for doc_id, doc_tf in dict_term_inverted_index.items():
                dot_product[doc_id] += query_tf * doc_tf


    sorted_docs = sorted(dot_product.items(), key=lambda item: item[1], reverse=True)
    if len(sorted_docs) < topk:
        return sorted_docs
    else:
        return sorted_docs[:topk]

def print_docs(dict_doc_list, list_doc):
    for doc_id, dist_doc in dict_doc_list:
        print(f"doc_id dot product is {dist_doc}")
        pp(list_doc[doc_id])
        print()

def extract_entities(text):
    doc = nlp(text)
    entities = []
    for ent in doc.ents:
        if len(ent.text.split()) > 1:
            entities.append(ent.text.lower())
    return entities



In [22]:
import os

# Check if we already have the preprocessed data
if os.path.exists('idf.pkl') and os.path.exists('doc_lengths.pkl'):
    print("Found existing files, loading")

    with open('idf.pkl', 'rb') as f:
        idf = pickle.load(f)

    with open('doc_lengths.pkl', 'rb') as f:
        doc_lengths = pickle.load(f)

    avg_doc_length = sum(doc_lengths) / len(doc_lengths)

    print(f"Loaded {len(idf)} IDF scores")
    print(f"Loaded {len(doc_lengths)} document lengths")
    print(f"Average doc length: {avg_doc_length:.2f}")

else:
    # Need to calculate everything from scratch
    print("Building IDF scores")
    num_docs = len(documents)
    idf = {}
    for term in inverted_index.keys():
        idf[term] = math.log((num_docs + 1) / (len(inverted_index[term]) + 1)) + 1
    print(f"Done - {len(idf)} terms processed")

    print("\nCalculating document lengths (takes a few minutes)")
    doc_lengths = [len(extract_terms(doc)) for doc in documents]
    avg_doc_length = sum(doc_lengths) / len(doc_lengths)
    print(f"Average doc length: {avg_doc_length:.2f}")

    # Save so we don't have to do this again
    print("\nSaving files")
    with open('idf.pkl', 'wb') as f:
        pickle.dump(idf, f)
    with open('doc_lengths.pkl', 'wb') as f:
        pickle.dump(doc_lengths, f)
    print("Saved successfully")

Found existing files, loading
Loaded 179862 IDF scores
Loaded 19994 document lengths
Average doc length: 227.13


In [23]:
# Save IDF and doc_lengths for future use
print("Saving IDF and doc_lengths")

with open('idf.pkl', 'wb') as f:
    pickle.dump(idf, f)

with open('doc_lengths.pkl', 'wb') as f:
    pickle.dump(doc_lengths, f)

print("Saved idf.pkl")
print("Saved doc_lengths.pkl")

Saving IDF and doc_lengths
Saved idf.pkl
Saved doc_lengths.pkl


In [24]:
# BM25 function from HW2
def compute_sim_query_bm25(query_terms, inverted_index, idf, topk=7, k1=1.5, b=0.75):
    """BM25 similarity - your best method from HW2"""
    query_term_freq = compute_freq(query_terms)
    document_scores = defaultdict(float)

    for query_term, query_tf in query_term_freq.items():
        if query_term in idf and query_term in inverted_index:
            idf_score = idf[query_term]
            for doc_id, doc_tf in inverted_index[query_term].items():
                numerator = doc_tf * (k1 + 1)
                denominator = doc_tf + k1 * (1 - b + b * (doc_lengths[doc_id] / avg_doc_length))
                term_score = idf_score * (numerator / denominator)
                document_scores[doc_id] += term_score

    sorted_docs = sorted(document_scores.items(), key=lambda item: item[1], reverse=True)
    return sorted_docs[:topk]

1.2
Show the top 7 results (e.g. movie title and plot) for each query using the tf-idf algorithm that gave you the best results in HW 2.

In [25]:
qlist = []
for i in range(len(queries)):
    qlist.append(extract_terms(queries[i]))

for j in range(len(qlist)):
    print(f"\n{'='*80}")
    print(f"QUERY {j+1}: {queries[j]}")
    print('='*80)

    results = compute_sim_query_bm25(qlist[j], inverted_index, idf, topk=7)

    for rank, (doc_id, scores) in enumerate(results, 1):
        title = post1980.iloc[doc_id]["Title"]
        plot = post1980.iloc[doc_id]["Plot"]
        print(f"\n{rank}. Title: {title}")
        print(f"   Plot: {plot[:200]}...")


QUERY 1: scary movies to watch at night

1. Title: Winter
   Plot: Jayaram and Bhavana takes the lead roles as Dr. Ramdas and his wife in the film. Dr. Ramdas is leading medical practitioner in Hyderabad. Fed up with the fast-paced city life, Ramdas and his wife deci...

2. Title: Dark Tales of Japan
   Plot: Introduction: Would You Like to Hear a Scary Tale? (Intorodakushon: Kowai hanashi, kikitai desu ka) Directed by Yoshihiro Nakamura; teleplay by Yoshihiro Nakamura and Katsuhide Suzuki
Plot: At a bus ...

3. Title: A Paying Ghost
   Plot: [3] The movie, “Paying Ghost”, is based on a comedy fiction by renowned Marathi novelist, Late P. V. Kale. It picturizes typical life of a Mumbai resident and how a ghost helps him to overcome the cha...

4. Title: Whore
   Plot: Liz is a Los Angeles street prostitute. The audience first sees her attempting to get a customer on a busy downtown street near a tunnel. She addresses the audience directly on her life and problems t...

5. Title: Pooh'

1.3 Select the relevant and not-relevant results in the top 7 (you already have this from HW 2)

In [26]:
q1_relevant = [18567, 14563, 12903]
q1_non_relevant = [6820, 17148, 7485, 4940]

q2_relevant = [14012, 17143, 19077, 19146, 19064]
q2_non_relevant = [12483, 13858]

q3_relevant = [18800, 16860, 14546, 18478]
q3_non_relevant = [16146, 17630, 11467]
#pulled from assignment 2 part 4

1.4
Apply the Rocchio algorithm (compute the modified query and then pass it through the same tf-idf function). Set alpha to 1 and try different sets of values for beta and gamma. Play with the number of terms you include in the modified query (variable th_rel and th_not_rel - see the function get_feedback_query in the feedback_retrieval_class.ipynb).

In [31]:
def get_feedback_query(orig_query, inverted_index, collection, ind_rel_doc, ind_not_rel, alpha, beta, gamma):
  # Start with the original query and scale by alpha
  mod_query = {term: alpha * tf for term, tf in orig_query.items()}

  size_vocab = len(inverted_index)

  # Add terms from relevant documents
  if ind_rel_doc:
    relevant_terms = defaultdict(float)

    for doc_id in ind_rel_doc:
      terms = extract_terms(collection[doc_id])
      term_freq = compute_freq(terms)

      for term, tf in term_freq.items():
        if term in inverted_index:
          doc_freq = len(inverted_index[term])
          tf_idf = tf * math.log((1 + size_vocab) / doc_freq)
          relevant_terms[term] += tf_idf

    # Get average across relevant docs
    num_rel = len(ind_rel_doc)
    for term, total_tfidf in relevant_terms.items():
      avg_tfidf = total_tfidf / num_rel
      if term in mod_query:
        mod_query[term] += beta * avg_tfidf
      else:
        mod_query[term] = beta * avg_tfidf

  # Subtract terms from non-relevant documents
  if ind_not_rel:
    nonrelevant_terms = defaultdict(float)

    for doc_id in ind_not_rel:
      terms = extract_terms(collection[doc_id])
      term_freq = compute_freq(terms)

      for term, tf in term_freq.items():
        if term in inverted_index:
          doc_freq = len(inverted_index[term])
          tf_idf = tf * math.log((1 + size_vocab) / doc_freq)
          nonrelevant_terms[term] += tf_idf

    # Get average across non-relevant docs
    num_not_rel = len(ind_not_rel)
    for term, total_tfidf in nonrelevant_terms.items():
      avg_tfidf = total_tfidf / num_not_rel
      if term in mod_query:
        mod_query[term] -= gamma * avg_tfidf
      else:
        mod_query[term] = -gamma * avg_tfidf

  # Remove any terms with negative weights
  mod_query = {term: weight for term, weight in mod_query.items() if weight > 0}

  return mod_query

In [32]:
import numpy as np

def precision_at_k(relevant_doc_ids, ranked_pairs, k):
    # Calculate precision at k for ranked results
    if k <= 0 or not ranked_pairs:
        return 0.0
    topk_ids = [doc_id for (doc_id, _) in ranked_pairs[:k]]
    hits = sum(1 for d in topk_ids if d in relevant_doc_ids)
    return hits / min(k, len(ranked_pairs))

def mod_query_to_string(mod_query_dict, th_rel=8, th_not_rel=0):
    # Convert modified query dict to string by keeping top weighted terms
    if not mod_query_dict:
        return ""

    # Sort terms by weight
    items = sorted(mod_query_dict.items(), key=lambda x: x[1], reverse=True)

    # Drop lowest weighted terms if needed
    if th_not_rel > 0 and len(items) > th_not_rel:
        items = items[:-th_not_rel]

    # Keep only top th_rel terms
    items = items[:max(1, th_rel)]
    terms = [t for t, _ in items]
    return " ".join(terms)

In [33]:
def run_rocchio_once(
    orig_query_str,
    rel_ids, nrel_ids,
    beta, gamma,
    th_rel=8, th_not_rel=0,
    topk=7
):
    # Get terms from original query
    q_terms = extract_terms(orig_query_str)
    q_tf = compute_freq(q_terms)

    # Run Rocchio algorithm
    mod_query_dict = get_feedback_query(
        orig_query=q_tf,
        inverted_index=inverted_index,
        collection=documents,
        ind_rel_doc=rel_ids,
        ind_not_rel=nrel_ids,
        alpha=1.0,
        beta=beta,
        gamma=gamma
    )

    # Convert modified query dict back to string
    mod_query_str = mod_query_to_string(mod_query_dict, th_rel=th_rel, th_not_rel=th_not_rel)

    # Run BM25 on the modified query
    ranked = compute_sim_query_bm25(extract_terms(mod_query_str), inverted_index, idf, topk=topk)

    return mod_query_str, ranked

In [34]:
import time

# Test different parameter combinations
beta_vals = [0.5, 0.75]
gamma_vals = [0.1, 0.3]
th_rel_vals = [8]
th_not_rel_vals = [0, 2]

rows = []
start_time = time.time()

# Group relevant and non-relevant docs for each query
rel_sets = [q1_relevant, q2_relevant, q3_relevant]
nrel_sets = [q1_non_relevant, q2_non_relevant, q3_non_relevant]

for qi, q in enumerate(queries):
    rel = rel_sets[qi]
    nrel = nrel_sets[qi]

    print(f"\nRunning query {qi+1}: {q}")
    for beta in beta_vals:
        for gamma in gamma_vals:
            for th_rel in th_rel_vals:
                for th_not_rel in th_not_rel_vals:
                    mod_q, ranked = run_rocchio_once(
                        orig_query_str=q,
                        rel_ids=rel, nrel_ids=nrel,
                        beta=beta, gamma=gamma,
                        th_rel=th_rel, th_not_rel=th_not_rel,
                        topk=7
                    )
                    p3 = precision_at_k(rel, ranked, 3)
                    p5 = precision_at_k(rel, ranked, 5)
                    p7 = precision_at_k(rel, ranked, 7)

                    rows.append({
                        "query_idx": qi+1,
                        "orig_query": q,
                        "modified_query": mod_q,
                        "beta": beta, "gamma": gamma,
                        "th_rel": th_rel, "th_not_rel": th_not_rel,
                        "P@3": round(p3, 3), "P@5": round(p5, 3), "P@7": round(p7, 3)
                    })

end_time = time.time()
print(f"\nFinished in {end_time - start_time:.1f} seconds ({len(rows)} total runs)\n")

df_results = pd.DataFrame(rows)
df_results.sort_values(["query_idx","P@7","P@5","P@3"], ascending=[True, False, False, False]).head(10)


Running query 1: scary movies to watch at night

Running query 2: romantic comedy movies to watch for fun

Running query 3: worst action movies of all time

Finished in 18.0 seconds (24 total runs)



,query_idx,orig_query,modified_query,beta,gamma,th_rel,th_not_rel,P@3,P@5,P@7
0,1,scary movies to watch at night,scary teleplay story yoshihiro kunal ghost chi...,0.50,0.1,8,0,1.0,0.6,0.429
1,1,scary movies to watch at night,scary teleplay story yoshihiro kunal ghost chi...,0.50,0.1,8,2,1.0,0.6,0.429
2,1,scary movies to watch at night,scary teleplay story yoshihiro kunal ghost ram...,0.50,0.3,8,0,1.0,0.6,0.429
3,1,scary movies to watch at night,scary teleplay story yoshihiro kunal ghost ram...,0.50,0.3,8,2,1.0,0.6,0.429
4,1,scary movies to watch at night,scary teleplay story yoshihiro kunal ghost chi...,0.75,0.1,8,0,1.0,0.6,0.429
5,1,scary movies to watch at night,scary teleplay story yoshihiro kunal ghost chi...,0.75,0.1,8,2,1.0,0.6,0.429
6,1,scary movies to watch at night,scary teleplay story yoshihiro kunal ghost ram...,0.75,0.3,8,0,1.0,0.6,0.429
7,1,scary movies to watch at night,scary teleplay story yoshihiro kunal ghost ram...,0.75,0.3,8,2,1.0,0.6,0.429
8,2,romantic comedy movies to watch for fun,kotoko naoki romantic comedy dad aihara ikeman...,0.50,0.1,8,0,1.0,0.6,0.571
9,2,romantic comedy movies to watch for fun,kotoko naoki romantic comedy dad aihara ikeman...,0.50,0.1,8,2,1.0,0.6,0.571


1.5 Compute the precision in top 3, top 5, and top 7 for what you think is the best result with Rocchio. Keep track of the experiments you did with the values of beta and gamma, th_rel and th_not_rel, and describe them in Answers.md

In [35]:
#1.5 before rocchio
def best_rows_by_query(df):
    best = []
    for qi in sorted(df["query_idx"].unique()):
        sub = df[df["query_idx"] == qi].sort_values(
            ["P@7","P@5","P@3"], ascending=[False, False, False]
        )
        if len(sub):
            best.append(sub.iloc[0])
    return pd.DataFrame(best)

best_df = best_rows_by_query(df_results)[
    ["query_idx","orig_query","modified_query","beta","gamma","th_rel","th_not_rel","P@3","P@5","P@7"]
]
best_df


,query_idx,orig_query,modified_query,beta,gamma,th_rel,th_not_rel,P@3,P@5,P@7
0,1,scary movies to watch at night,scary teleplay story yoshihiro kunal ghost chi...,0.5,0.1,8,0,1.000,0.6,0.429
8,2,romantic comedy movies to watch for fun,kotoko naoki romantic comedy dad aihara ikeman...,0.5,0.1,8,0,1.000,0.6,0.571
16,3,worst action movies of all time,shin chan action mask prabhakaran lemon shinno...,0.5,0.1,8,0,0.667,0.4,0.429


In [37]:
# 1.5 - Top 7 results after Rocchio feedback

print("\n" + "="*80)
print("TOP 7 MOVIES AFTER ROCCHIO FEEDBACK")
print("="*80)

for idx, row in best_df.iterrows():
    qi = int(row['query_idx']) - 1

    print(f"\n{'='*80}")
    print(f"QUERY {qi+1}: {queries[qi]}")
    print(f"Modified Query: {row['modified_query']}")
    print(f"Parameters: beta={row['beta']}, gamma={row['gamma']}, th_rel={row['th_rel']}, th_not_rel={row['th_not_rel']}")
    print(f"Precision: P@3={row['P@3']}, P@5={row['P@5']}, P@7={row['P@7']}")
    print('='*80)

    rel_sets = [q1_relevant, q2_relevant, q3_relevant]
    nrel_sets = [q1_non_relevant, q2_non_relevant, q3_non_relevant]

    # Run Rocchio with best parameters
    mod_q_str, ranked = run_rocchio_once(
        orig_query_str=queries[qi],
        rel_ids=rel_sets[qi],
        nrel_ids=nrel_sets[qi],
        beta=row['beta'],
        gamma=row['gamma'],
        th_rel=int(row['th_rel']),
        th_not_rel=int(row['th_not_rel']),
        topk=7
    )

    for rank, (doc_id, score) in enumerate(ranked, 1):
        title = post1980.iloc[doc_id]["Title"]
        plot = post1980.iloc[doc_id]["Plot"]

        relevance = "[RELEVANT]" if doc_id in rel_sets[qi] else ""

        print(f"\n{rank}. {title} {relevance}")
        print(f"   Score: {score:.4f}")
        print(f"   Plot: {plot[:200]}...")


TOP 7 MOVIES AFTER ROCCHIO FEEDBACK

QUERY 1: scary movies to watch at night
Modified Query: scary teleplay story yoshihiro kunal ghost child ramdas
Parameters: beta=0.5, gamma=0.1, th_rel=8, th_not_rel=0
Precision: P@3=1.0, P@5=0.6, P@7=0.429

1. Dark Tales of Japan [RELEVANT]
   Score: 59.4123
   Plot: Introduction: Would You Like to Hear a Scary Tale? (Intorodakushon: Kowai hanashi, kikitai desu ka) Directed by Yoshihiro Nakamura; teleplay by Yoshihiro Nakamura and Katsuhide Suzuki
Plot: At a bus ...

2. Winter [RELEVANT]
   Score: 37.4349
   Plot: Jayaram and Bhavana takes the lead roles as Dr. Ramdas and his wife in the film. Dr. Ramdas is leading medical practitioner in Hyderabad. Fed up with the fast-paced city life, Ramdas and his wife deci...

3. Darna Zaroori Hai [RELEVANT]
   Score: 35.2701
   Plot: Darna Zaroori Hai interweaves six stories into one film. Five children get lost in the middle of a forest until they find a haunted house. Inside, there is an old woman who agre

Expand your query with synonyms and thesaurus words.

Use the same queries as before, but expand each of them with additional words (e.g., synonyms, thesaurus) from your collection vocabulary. See the example in gen_synonyms.ipynb in the GitHub repository. It uses glove embedding and/or WordNet to find synonyms. First, extract the nouns from your query, then find synonyms for each noun. Finally, add all synonyms to the original query, which will become your modified query.
Apply the same tf-idf algorithm you used on one to the new, modified query. Show the top 7 results for each query.
Compute precision in top 3, top 5, and top 7.

In [ ]:
#part 2
import nltk
from nltk.corpus import wordnet as wn
from nltk.corpus.reader.wordnet import WordNetError
nltk.download('wordnet')
nltk.download('averaged_perceptron_tagger')
nltk.download('omw-1.4')





[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...


True

In [ ]:
def expand_query_with_synonyms(query_str):
    # Expands a query string with synonyms of its nouns using WordNet.

    # Returns:
    # A string representing the expanded query.

    doc = nlp(query_str)
    original_terms = [token.text.lower() for token in doc if token.is_alpha and not token.is_stop]
    synonyms = set()

    for token in doc:
        if token.pos_ == 'NOUN':
            # Get the WordNet part-of-speech tag
            if token.tag_.startswith('N'):
                wn_pos = wn.NOUN
            elif token.tag_.startswith('V'):
                wn_pos = wn.VERB
            elif token.tag_.startswith('R'):
                wn_pos = wn.ADV
            elif token.tag_.startswith('J'):
                wn_pos = wn.ADJ
            else:
                wn_pos = None

            if wn_pos:
                for syn in wn.synsets(token.text, pos=wn_pos):
                    for lemma in syn.lemmas():
                        synonyms.add(lemma.name().replace('_', ' ')) # Add synonyms, replace underscores with spaces

    expanded_terms = list(set(original_terms + list(synonyms))) # Combine original terms and unique synonyms
    return " ".join(expanded_terms)

In [ ]:
expanded_queries = []
for query in queries:
    expanded_query = expand_query_with_synonyms(query)
    expanded_queries.append(expanded_query)

print("Expanded Queries:")
for i, eq in enumerate(expanded_queries):
    print(f"Query {i+1}: {eq}")

Expanded Queries:
Query 1: film watch picture show movie dark moving-picture show nighttime scary Nox motion-picture show Night picture night moving picture pic movies flick motion picture
Query 2: film romantic merriment playfulness watch fun picture show funniness clowning moving-picture show drollery moving picture pic flick sport motion picture comedy movie play motion-picture show picture movies
Query 3: film action activeness natural process clock time legal action prison term sentence military action action mechanism natural action action at law activity picture show moving-picture show metre fourth dimension meter moving picture pic time flick motion picture movie worst clip motion-picture show picture movies


In [ ]:
expanded_query_results = []

for i, expanded_query_str in enumerate(expanded_queries):
    print(f"\n{'='*80}")
    print(f"EXPANDED QUERY {i+1}: {expanded_query_str}")
    print('='*80)

    # Use the same BM25 retrieval function
    expanded_query_terms = extract_terms(expanded_query_str)
    ranked_docs = compute_sim_query_bm25(expanded_query_terms, inverted_index, idf, topk=7)

    # Get relevant document IDs for the original query to compute precision
    if i == 0:
        rel_ids = q1_relevant
    elif i == 1:
        rel_ids = q2_relevant
    else:
        rel_ids = q3_relevant

    p3 = precision_at_k(set(rel_ids), ranked_docs, 3)
    p5 = precision_at_k(set(rel_ids), ranked_docs, 5)
    p7 = precision_at_k(set(rel_ids), ranked_docs, 7)

    print(f"\nPrecision@3: {p3:.3f}")
    print(f"Precision@5: {p5:.3f}")
    print(f"Precision@7: {p7:.3f}")

    expanded_query_results.append({
        "query_idx": i+1,
        "expanded_query": expanded_query_str,
        "ranked_docs": ranked_docs,
        "P@3": round(p3, 3),
        "P@5": round(p5, 3),
        "P@7": round(p7, 3)
    })

    # Display top results (movie title and plot)
    print("\nTop 7 Results:")
    for rank, (doc_id, scores) in enumerate(ranked_docs, 1):
        title = post1980.iloc[doc_id]["Title"]
        plot = post1980.iloc[doc_id]["Plot"]
        print(f"\n{rank}. Title: {title}")
        print(f"   Plot: {plot[:200]}...")


EXPANDED QUERY 1: film watch picture show movie dark moving-picture show nighttime scary Nox motion-picture show Night picture night moving picture pic movies flick motion picture

Precision@3: 0.667
Precision@5: 0.400
Precision@7: 0.286

Top 7 Results:

1. Title: Dark Tales of Japan
   Plot: Introduction: Would You Like to Hear a Scary Tale? (Intorodakushon: Kowai hanashi, kikitai desu ka) Directed by Yoshihiro Nakamura; teleplay by Yoshihiro Nakamura and Katsuhide Suzuki
Plot: At a bus ...

2. Title: Ballistic Kiss
   Plot: Cat (Donnie Yen) is an aimless killer, no matter if its raining or sunny, he would always wear dark glasses and always calls to the radio to talk. Every night, he looks out the window to see his angel...

3. Title: Winter
   Plot: Jayaram and Bhavana takes the lead roles as Dr. Ramdas and his wife in the film. Dr. Ramdas is leading medical practitioner in Hyderabad. Fed up with the fast-paced city life, Ramdas and his wife deci...

4. Title: Bayam Oru Payanam
   

In [ ]:
df_expanded_results = pd.DataFrame(expanded_query_results)
df_expanded_results[['query_idx', 'expanded_query', 'P@3', 'P@5', 'P@7']]

,query_idx,expanded_query,P@3,P@5,P@7
0,1,film watch picture show movie dark moving-pict...,0.667,0.4,0.286
1,2,film romantic merriment playfulness watch fun ...,0.333,0.6,0.429
2,3,film action activeness natural process clock t...,0.000,0.0,0.000
